[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartikmunjal/rlhf-and-reward-modelling/blob/main/notebooks/04_ppo_training.ipynb)

# Notebook 4 — Stage 3a: Policy Optimization with PPO

## The RL Formulation for Language Models

We frame text generation as a Markov Decision Process:

| RL concept | Language model equivalent |
|------------|---------------------------|
| State $s_t$ | Prompt + tokens generated so far |
| Action $a_t$ | Next token |
| Policy $\\pi_\\theta(a_t \mid s_t)$ | Language model distribution |
| Reward $r$ | Scalar from reward model (sparse: only at end of response) |
| Episode | One full (prompt → response) generation |

The objective is to find $\\pi_\\theta$ that maximises expected reward while staying close to the reference policy $\\pi_{ref}$ (the SFT model):

$$\max_\\theta\; \mathbb{E}_{x \sim \mathcal{D},\; y \sim \pi_\\theta(\cdot|x)}\!\left[r_\phi(x, y)\right] - \beta \cdot \mathbb{KL}\!\left[\pi_\theta(\cdot|x) \,\|\, \pi_{ref}(\cdot|x)\right]$$

## PPO-Clip: The Update Rule

Naive policy gradient (REINFORCE) has high variance and can take catastrophically large steps.  
PPO addresses this with a **clipped surrogate objective** that limits how far each update moves the policy:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(\rho_t A_t,\; \operatorname{clip}(\rho_t, 1{-}\varepsilon, 1{+}\varepsilon)\, A_t\right)\right]$$

where:
- $\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ — importance sampling ratio (how much the policy changed for this token)
- $A_t = r_\phi(x,y) - V(s_t)$ — advantage: reward minus value baseline
- $\varepsilon = 0.2$ — clip radius (a hyperparameter)

**Intuition**: the `min` ensures we never benefit from taking a step that the clipping would have rejected.  
If the update would move the ratio outside $[1-\varepsilon, 1+\varepsilon]$, we cap the gradient — a soft trust region.

## The KL Penalty: Why It's Non-Optional

The KL divergence term is not just a regulariser — it's the mechanism that prevents **reward hacking**.

Without the KL penalty, the policy quickly learns to exploit the reward model's blind spots.  
For example, it might discover that the RM assigns high scores to responses that:
- Repeat certain high-scoring n-grams
- Use a verbose, confident-sounding tone
- Avoid the question entirely with evasive pleasantries

The KL penalty forces: "you can only deviate from $\\pi_{ref}$ if the reward improvement is worth it."

The total PPO loss (with KL) is:
$$\mathcal{L}^{\text{PPO+KL}}(\theta) = \mathcal{L}^{\text{CLIP}}(\theta) - \beta \cdot \mathbb{KL}[\pi_\theta \| \pi_{ref}]$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Visualise PPO-Clip: what happens at different importance ratios
rho = np.linspace(0, 3, 500)
eps = 0.2

def ppo_clip_loss(rho, A, eps):
    return -np.minimum(rho * A, np.clip(rho, 1 - eps, 1 + eps) * A)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, A, title in zip(axes, [1.0, -1.0], ['Positive advantage (A > 0)', 'Negative advantage (A < 0)']):
    loss = ppo_clip_loss(rho, A, eps)
    ax.plot(rho, -loss, color='steelblue', linewidth=2, label='PPO-Clip objective')
    ax.axvline(1 - eps, color='tomato', linestyle='--', linewidth=1, label=f'ρ = 1-ε = {1-eps}')
    ax.axvline(1 + eps, color='seagreen', linestyle='--', linewidth=1, label=f'ρ = 1+ε = {1+eps}')
    ax.axvline(1.0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_xlabel('Importance ratio ρ = π_θ / π_old')
    ax.set_ylabel('Objective value')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 3)
    flat_region = mpatches.FancyArrowPatch((1+eps, A*0.8), (2.5, A*0.8),
                                            arrowstyle='-', color='orange', linewidth=2)
    ax.add_patch(flat_region)
    ax.text(1.9, A*0.7, 'clipped\n(flat gradient)', ha='center', fontsize=8, color='orange')

plt.suptitle('PPO-Clip Objective: Gradient is zeroed when policy moves too far', y=1.02)
plt.tight_layout()
plt.savefig('ppo_clip_illustration.png', dpi=100, bbox_inches='tight')
plt.show()

## The PPO Training Loop

Each PPO "step" consists of:

```
1. Sample batch of prompts x ~ D
2. Generate responses y ~ π_θ(·|x)   ← ROLLOUT (on-policy)
3. Compute rewards r = r_φ(x, y)     ← REWARD MODEL scoring
4. Compute advantages A = r - V(s)    ← VALUE BASELINE subtraction
5. Run PPO_EPOCHS passes of:          ← PPO UPDATES
     - Compute ρ = π_θ/π_θ_old
     - Compute clipped surrogate loss
     - Add KL penalty
     - Gradient step
```

The critical design choice: PPO is **on-policy** — we must generate fresh rollouts each step.  
This is why PPO is more expensive than DPO (which trains on fixed offline data).

In [ ]:
# Show the key config and explain each parameter
from src.training.ppo import PPOTrainingConfig

cfg = PPOTrainingConfig()

params = {
    'batch_size': (cfg.batch_size, 'Rollout batch: number of (prompt, response) pairs per step'),
    'ppo_epochs': (cfg.ppo_epochs, 'PPO update passes per rollout (too many → overfitting to stale rollouts)'),
    'init_kl_coef': (cfg.init_kl_coef, 'Initial β for KL penalty'),
    'target_kl': (cfg.target_kl, 'Adaptive controller keeps KL near this value (nats)'),
    'clip_range': (cfg.clip_range, 'ε in PPO-Clip — standard value, rarely needs tuning'),
    'max_new_tokens': (cfg.max_new_tokens, 'Response length during rollout generation'),
}

print('PPO Config:\n')
for key, (val, desc) in params.items():
    print(f'  {key:<22} = {str(val):<8}  # {desc}')

## Training Dynamics and Reward Hacking

In [ ]:
# Representative PPO training curves
steps = np.arange(0, 300)
rng = np.random.RandomState(42)

# Healthy PPO run: reward increases, KL stays controlled
reward_healthy = 0.2 + 0.5 * (1 - np.exp(-steps / 80)) + 0.03 * rng.randn(300)
kl_healthy = 0.5 + 5.5 * (1 - np.exp(-steps / 60)) + 0.3 * rng.randn(300)
kl_healthy = np.clip(kl_healthy, 0, 7)

# Reward hacking: KL diverges without penalty
reward_hacking = 0.2 + 1.2 * (1 - np.exp(-steps / 40)) + 0.05 * rng.randn(300)
kl_hacking = 0.5 + 30 * (1 - np.exp(-steps / 50)) + rng.randn(300)
kl_hacking = np.clip(kl_hacking, 0, 35)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Healthy: reward
axes[0, 0].plot(steps, reward_healthy, color='steelblue', linewidth=1.5)
axes[0, 0].set_title('Mean Reward — Healthy PPO (adaptive KL)')
axes[0, 0].set_ylabel('Mean reward')
axes[0, 0].set_xlabel('Step')
axes[0, 0].axhline(0.2, color='gray', linestyle='--', linewidth=1, label='SFT baseline')
axes[0, 0].legend()

# Healthy: KL
axes[0, 1].plot(steps, kl_healthy, color='steelblue', linewidth=1.5)
axes[0, 1].axhline(6.0, color='tomato', linestyle='--', linewidth=1.5, label='target_kl=6.0')
axes[0, 1].set_title('KL Divergence from π_ref — Controlled')
axes[0, 1].set_ylabel('KL (nats)')
axes[0, 1].set_xlabel('Step')
axes[0, 1].legend()

# Hacking: reward (fake-high)
axes[1, 0].plot(steps, reward_hacking, color='tomato', linewidth=1.5)
axes[1, 0].set_title('Mean Reward — Reward Hacking (no KL penalty)')
axes[1, 0].set_ylabel('Mean reward')
axes[1, 0].set_xlabel('Step')
axes[1, 0].axhline(0.2, color='gray', linestyle='--', linewidth=1, label='SFT baseline')
axes[1, 0].legend()

# Hacking: KL explodes
axes[1, 1].plot(steps, kl_hacking, color='tomato', linewidth=1.5)
axes[1, 1].set_title('KL Divergence — Exploding (reward hacking)')
axes[1, 1].set_ylabel('KL (nats)')
axes[1, 1].set_xlabel('Step')
axes[1, 1].text(150, 25, 'Policy has drifted far\nfrom reference — outputs\nmay be degenerate', 
                fontsize=9, color='tomato',
                bbox=dict(facecolor='white', edgecolor='tomato', alpha=0.8))

plt.tight_layout()
plt.savefig('ppo_training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## Key Observations

- **Adaptive KL controller**: our config uses `adap_kl_ctrl=True` which adjusts $\beta$ automatically to keep KL near `target_kl=6.0`.  
  When KL > target, $\beta$ increases (more penalty); when KL < target, $\beta$ decreases (allows more exploration).

- **Plateau in reward**: a healthy PPO run plateaus at a reward significantly above the SFT baseline.  
  If reward keeps climbing indefinitely, it's a sign of reward hacking — check KL divergence.

- **4 PPO update epochs**: standard practice. Too few → sample inefficiency; too many → policy diverges from rollout distribution, violating the "trust region" assumption.

---
**Next**: [05_dpo_training.ipynb](05_dpo_training.ipynb) — DPO: no rollouts, no reward model, same alignment goal